# 指示文の言語を変えて LLM の出力がどう変わるか比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_llm/prompt_language.ipynb)

NLP クックブック 21-09 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

日本語のデータを LLM に処理させるとき、指示文（プロンプト）を日本語で書くのと英語で書くのとで、結果が変わるかを確かめたい。

In [1]:
!pip install -q datasets

In [2]:
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

ds = load_dataset("sbintuitions/JCommonsenseQA", split="validation").shuffle(seed=0).select(range(150))     # 5 択の常識問題
print(ds[0])

@torch.no_grad()
def choose(prompt, n_choices=5):
    """選択肢の番号 (1〜5) のうち、次に出る確率が最も高いものを答えとする"""
    digits = [tokenizer(str(i), add_special_tokens=False)["input_ids"][-1] for i in range(1, n_choices + 1)]
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(llm.device)
    return int(llm(ids).logits[0, -1, digits].argmax())

{'q_id': 9279, 'question': '福井県の南西部、若狭のほぼ中央に位置する市は？', 'choice0': '小鍋', 'choice1': '小浜', 'choice2': '大浜', 'choice3': '小売', 'choice4': '高浜', 'label': 1}


In [3]:
def body(ex):
    return "\n".join(f"{i + 1}. {ex[f'choice{i}']}" for i in range(5))

PROMPTS = {
    "日本語の指示": lambda ex: f"次の質問に対して、最も適切な選択肢の番号を 1 つだけ答えてください。\n\n質問: {ex['question']}\n{body(ex)}\n\n答えの番号:",
    "英語の指示": lambda ex: f"Answer the following question by choosing the most appropriate option. Reply with the number only.\n\nQuestion: {ex['question']}\n{body(ex)}\n\nAnswer:",
    "日本語の指示 (言い回しを変える)": lambda ex: f"{ex['question']}\n\n選択肢:\n{body(ex)}\n\n正しいものを選び、番号だけで答えよ。",
    "英語の指示 (言い回しを変える)": lambda ex: f"{ex['question']}\n\nOptions:\n{body(ex)}\n\nWhich option is correct? Output only its number.",
}
predictions = {label: [choose(make(ex)) for ex in ds] for label, make in PROMPTS.items()}
gold = np.array(ds["label"])
print(pd.Series({label: (np.array(p) == gold).mean() for label, p in predictions.items()}).round(3).to_string())

日本語の指示               0.873
英語の指示                0.887
日本語の指示 (言い回しを変える)    0.880
英語の指示 (言い回しを変える)     0.880


In [4]:
# 差は偶然の範囲か: ブートストラップで正解率の差の信頼区間を求める
rng = np.random.default_rng(0)
def diff_ci(a, b, n=2000):
    a, b = np.array(predictions[a]) == gold, np.array(predictions[b]) == gold
    idx = rng.integers(0, len(gold), (n, len(gold)))
    d = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    return round(float(a.mean() - b.mean()), 3), np.percentile(d, [2.5, 97.5]).round(3).tolist()

print("日本語 − 英語:", diff_ci("日本語の指示", "英語の指示"))
print("日本語どうし (言い回しの違い):", diff_ci("日本語の指示", "日本語の指示 (言い回しを変える)"))
print("英語どうし (言い回しの違い):", diff_ci("英語の指示", "英語の指示 (言い回しを変える)"))
agree = pd.DataFrame({a: {b: (np.array(predictions[a]) == np.array(predictions[b])).mean() for b in predictions} for a in predictions})
print("予測の一致率:"); print(agree.round(2).to_string())

日本語 − 英語: (-0.013, [-0.053, 0.02])
日本語どうし (言い回しの違い): (-0.007, [-0.04, 0.027])
英語どうし (言い回しの違い): (0.007, [-0.027, 0.04])
予測の一致率:
                   日本語の指示  英語の指示  日本語の指示 (言い回しを変える)  英語の指示 (言い回しを変える)
日本語の指示               1.00   0.95               0.94              0.94
英語の指示                0.95   1.00               0.93              0.93
日本語の指示 (言い回しを変える)    0.94   0.93               1.00              0.99
英語の指示 (言い回しを変える)     0.94   0.93               0.99              1.00


- 多言語の LLM では「指示は英語のほうが良い」と言われることがあるが、モデルとタスクによる。日本語を中心に学習されたモデルでは、日本語の指示のほうが良いか、差がないことが多い。ここでも英語の指示のほうが 1 ポイントほど高いが、差の信頼区間は 0 をまたいでおり、差があるとは言えない（T4 と同じ float16 で実行した値）。
- 言語の違いによる差が、同じ言語の中での言い回しの違いによる差より大きいかどうかを見る。言い回しを変えただけで同じくらい変動するなら、「言語の効果」とは言えない。
- 公平に比べるのは難しい。英語の指示を機械翻訳して日本語にすると、不自然な日本語（翻訳調）になり、日本語側が不利になる。それぞれの言語で自然に書いた指示どうしを比べる。
- 150 問程度では、数ポイントの差は誤差の範囲に入る。信頼区間が 0 をまたいでいたら、差があるとは言えない。信頼区間の求め方は評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照。
- 出力の言語にも影響する。英語で指示すると、日本語の入力に対しても英語で答え始めることがある。出力の言語を明示する。